# Assignment 1 - Multimodal Classification

## 4.3 Với tập dữ liệu đa phương thức

So sánh hai cách tiếp cận:
- **Zero-shot classification**: Sử dụng CLIP pretrained để phân loại mà không cần fine-tune
- **Few-shot classification**: Fine-tune CLIP với một số lượng nhỏ mẫu huấn luyện

**Dataset**: COCO (Common Objects in Context) - public version, no authentication required

**Model**: CLIP (Contrastive Language-Image Pre-training)

## 1. Import Libraries và Setup

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import time
import random
import warnings
from typing import List, Dict, Tuple
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

# ── Data Manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# ── Transformers & CLIP ───────────────────────────────────────────────────────
from transformers import CLIPProcessor, CLIPModel
from transformers import get_linear_schedule_with_warmup

# ── Sklearn Metrics ───────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

# ── COCO Tools ────────────────────────────────────────────────────────────────
import json
from pycocotools.coco import COCO

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Device Configuration ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load và Chuẩn bị COCO Dataset (Kaggle)

Sử dụng COCO dataset từ Kaggle. Trên Kaggle, COCO dataset được lưu tại `/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/`.

**COCO 2017 Structure**:
- Train images: `/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017/`
- Validation images: `/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017/`
- Annotations: `/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/`

Chúng ta sẽ tạo classification task dựa trên các object categories phổ biến.

In [ ]:
# Kaggle COCO paths
KAGGLE_COCO_ROOT = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
TRAIN_IMG_DIR = f"{KAGGLE_COCO_ROOT}/train2017"
VAL_IMG_DIR = f"{KAGGLE_COCO_ROOT}/val2017"
TRAIN_ANN_FILE = f"{KAGGLE_COCO_ROOT}/annotations/instances_train2017.json"
VAL_ANN_FILE = f"{KAGGLE_COCO_ROOT}/annotations/instances_val2017.json"
CAPTIONS_TRAIN_FILE = f"{KAGGLE_COCO_ROOT}/annotations/captions_train2017.json"
CAPTIONS_VAL_FILE = f"{KAGGLE_COCO_ROOT}/annotations/captions_val2017.json"

# Check if running on Kaggle
IS_KAGGLE = os.path.exists(KAGGLE_COCO_ROOT)

if IS_KAGGLE:
    print("✓ Running on Kaggle. Using Kaggle COCO dataset.")
    print(f"  Root: {KAGGLE_COCO_ROOT}")
else:
    print("⚠️  Not running on Kaggle!")
    print("   Please update KAGGLE_COCO_ROOT to your local COCO path.")

# Define target categories (COCO standard categories)
# COCO category IDs: 1=person, 3=car, 62=chair, 84=book, 44=bottle,
#                    18=dog, 17=cat, 16=bird, 19=horse, 5=airplane
TARGET_CATEGORIES = {
    1: 'person',
    3: 'car',
    62: 'chair',
    84: 'book',
    44: 'bottle',
    18: 'dog',
    17: 'cat',
    16: 'bird',
    19: 'horse',
    5: 'airplane'
}

TARGET_CATEGORY_IDS = list(TARGET_CATEGORIES.keys())
TARGET_CATEGORY_NAMES = list(TARGET_CATEGORIES.values())

print(f"\nTarget categories: {TARGET_CATEGORY_NAMES}")
print(f"Number of classes: {len(TARGET_CATEGORY_NAMES)}")

In [ ]:
def load_coco_kaggle(img_dir, ann_file, captions_file, target_cat_ids, max_samples=5000):
    """
    Load COCO dataset from Kaggle directory structure.
    """
    print(f"\nLoading COCO from Kaggle...")
    print(f"  Images: {img_dir}")
    print(f"  Annotations: {ann_file}")
    print(f"  Captions: {captions_file}")
    
    # Initialize COCO API
    coco = COCO(ann_file)
    coco_caps = COCO(captions_file)
    
    # Get image IDs that contain target categories
    img_ids_set = set()
    for cat_id in target_cat_ids:
        img_ids = coco.getImgIds(catIds=[cat_id])
        img_ids_set.update(img_ids)
    
    print(f"  Found {len(img_ids_set)} images with target categories")
    
    # Filter and collect samples
    filtered_data = []
    category_counts = Counter()
    samples_per_class = int(max_samples / len(target_cat_ids) * 1.5)
    
    for img_id in img_ids_set:
        if len(filtered_data) >= max_samples:
            break
        
        # Get image info
        img_info = coco.loadImgs([img_id])[0]
        img_path = os.path.join(img_dir, img_info['file_name'])
        
        # Check if image exists
        if not os.path.exists(img_path):
            continue
        
        # Get annotations for this image
        ann_ids = coco.getAnnIds(imgIds=[img_id], catIds=target_cat_ids)
        anns = coco.loadAnns(ann_ids)
        
        if not anns:
            continue
        
        # Get primary category (most prominent object by area)
        primary_cat_id = max(anns, key=lambda x: x.get('area', 0))['category_id']
        
        if primary_cat_id not in target_cat_ids:
            continue
        
        category_name = TARGET_CATEGORIES[primary_cat_id]
        
        # Check if we already have enough samples for this category
        if category_counts[category_name] >= samples_per_class:
            continue
        
        # Get caption
        cap_ids = coco_caps.getAnnIds(imgIds=[img_id])
        captions = coco_caps.loadAnns(cap_ids)
        caption = captions[0]['caption'] if captions else f"image containing {category_name}"
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            continue
        
        # Add to filtered data
        filtered_data.append({
            'image': image,
            'caption': caption,
            'category': category_name,
            'label': TARGET_CATEGORY_NAMES.index(category_name),
            'image_id': img_id,
            'file_name': img_info['file_name']
        })
        
        category_counts[category_name] += 1
        
        if len(filtered_data) % 500 == 0:
            print(f"  Collected {len(filtered_data)} samples")
            print(f"    Distribution: {dict(category_counts)}")
    
    print(f"\n✓ Loaded {len(filtered_data)} samples")
    print(f"\nCategory distribution:")
    for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
        print(f"  {cat}: {count}")
    
    return filtered_data, category_counts

# Load dataset from train and val splits
train_data, train_counts = load_coco_kaggle(
    TRAIN_IMG_DIR, TRAIN_ANN_FILE, CAPTIONS_TRAIN_FILE, 
    TARGET_CATEGORY_IDS, max_samples=4000
)

val_data, val_counts = load_coco_kaggle(
    VAL_IMG_DIR, VAL_ANN_FILE, CAPTIONS_VAL_FILE, 
    TARGET_CATEGORY_IDS, max_samples=1500
)

# Combine datasets
filtered_data = train_data + val_data
category_counts = train_counts + val_counts

print(f"\n✓ Combined dataset size: {len(filtered_data)}")
print(f"\nFinal distribution:")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}")

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(filtered_data)

# Remove categories with too few samples
min_samples = 100
category_counts_filtered = df['category'].value_counts()
valid_categories = category_counts_filtered[category_counts_filtered >= min_samples].index.tolist()

if len(valid_categories) < len(TARGET_CATEGORY_NAMES):
    print(f"\nFiltering out categories with < {min_samples} samples...")
    removed = set(TARGET_CATEGORY_NAMES) - set(valid_categories)
    print(f"Removed: {removed}")
    print(f"Keeping {len(valid_categories)} categories: {valid_categories}")
    df = df[df['category'].isin(valid_categories)].reset_index(drop=True)
    TARGET_CATEGORY_NAMES = valid_categories
    # Reassign labels
    df['label'] = df['category'].apply(lambda x: TARGET_CATEGORY_NAMES.index(x))

print(f"\n✓ Final DataFrame shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFinal label distribution:")
print(df['category'].value_counts())
print(f"\nSample data:")
print(df[['caption', 'category', 'label']].head(10))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar plot
df['category'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
df['category'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Caption length analysis
df['caption_length'] = df['caption'].apply(lambda x: len(str(x).split()))
print(f"\nCaption length statistics:")
print(df['caption_length'].describe())

In [ ]:
# Display sample images with captions
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

sample_indices = df.sample(min(8, len(df)), random_state=SEED).index
for idx, sample_idx in enumerate(sample_indices):
    img = df.iloc[sample_idx]['image']
    caption = str(df.iloc[sample_idx]['caption'])
    category = df.iloc[sample_idx]['category']
    
    # Convert to RGB if needed
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    axes[idx].imshow(img)
    axes[idx].set_title(f"{category}\n{caption[:50]}...", fontsize=9)
    axes[idx].axis('off')

# Hide empty subplots
for idx in range(len(sample_indices), 8):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 4. Dataset Split

Chia dữ liệu thành:
- **Test set**: 20% (để đánh giá cuối cùng)
- **Zero-shot**: Không dùng training data, chỉ dùng test set
- **Few-shot**: Dùng một số lượng nhỏ samples (k-shot per class) để fine-tune

In [ ]:
# Split into train and test
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=SEED, 
    stratify=df['label']
)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

# Create few-shot datasets with different k values
K_SHOTS = [1, 5, 10, 20]  # Number of samples per class

few_shot_datasets = {}
for k in K_SHOTS:
    few_shot_samples = []
    for category in TARGET_CATEGORY_NAMES:
        category_samples = train_df[train_df['category'] == category].sample(
            n=min(k, len(train_df[train_df['category'] == category])), 
            random_state=SEED
        )
        few_shot_samples.append(category_samples)
    
    few_shot_df = pd.concat(few_shot_samples).reset_index(drop=True)
    few_shot_datasets[k] = few_shot_df
    print(f"{k}-shot dataset size: {len(few_shot_df)} ({k} samples per class)")

print(f"\nTest set distribution:")
print(test_df['category'].value_counts())

## 5. Load CLIP Model và Processor

In [ ]:
# Load CLIP model
MODEL_NAME = "openai/clip-vit-base-patch32"

print(f"Loading CLIP model: {MODEL_NAME}")
clip_model = CLIPModel.from_pretrained(MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)

clip_model = clip_model.to(device)

# Model info
total_params = sum(p.numel() for p in clip_model.parameters())
trainable_params = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)

print(f"\nModel loaded: {MODEL_NAME}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Custom Dataset Class

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, dataframe, processor, target_categories):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.target_categories = target_categories
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = row['image']
        caption = str(row['caption'])
        label = row['label']
        
        # Convert to RGB if needed
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        return {
            'image': image,
            'caption': caption,
            'label': label
        }

def collate_fn(batch, processor):
    """
    Custom collate function to handle variable-length text and images.
    """
    images = [item['image'] for item in batch]
    captions = [item['caption'] for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    
    # Process all images and captions together for proper batching
    inputs = processor(
        text=captions,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )
    
    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'pixel_values': inputs['pixel_values'],
        'label': labels
    }

# Create custom collate function with processor
from functools import partial
collate_with_processor = partial(collate_fn, processor=clip_processor)

# Create test dataset
test_dataset = MultimodalDataset(test_df, clip_processor, TARGET_CATEGORY_NAMES)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_with_processor)

print(f"Test dataset size: {len(test_dataset)}")
print(f"Test batches: {len(test_loader)}")

## 7. Zero-Shot Classification

CLIP được huấn luyện để align image và text representations. Chúng ta có thể sử dụng nó trực tiếp để phân loại bằng cách so sánh image với text prompts của các categories.

In [ ]:
def zero_shot_classification(model, processor, test_df, target_categories, device):
    """
    Perform zero-shot classification using CLIP.
    """
    model.eval()
    
    # Create text prompts for each category
    text_prompts = [f"a photo of a {category}" for category in target_categories]
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    print("Performing zero-shot classification...")
    print(f"Text prompts: {text_prompts}")
    
    with torch.no_grad():
        for idx, row in test_df.iterrows():
            image = row['image']
            label = row['label']
            
            # Convert to RGB if needed
            if image.mode != 'RGB':
                image = image.convert('RGB')
            
            # Process inputs
            inputs = processor(
                text=text_prompts,
                images=image,
                return_tensors="pt",
                padding=True
            )
            
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Get model outputs
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)
            
            pred = probs.argmax(dim=1).item()
            
            all_preds.append(pred)
            all_labels.append(label)
            all_probs.append(probs.cpu().numpy()[0])
            
            if (idx + 1) % 100 == 0:
                print(f"Processed {idx + 1}/{len(test_df)} samples")
    
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

# Perform zero-shot classification
print("\n" + "="*80)
print("ZERO-SHOT CLASSIFICATION")
print("="*80)

start_time = time.time()
zero_shot_preds, zero_shot_labels, zero_shot_probs = zero_shot_classification(
    clip_model, clip_processor, test_df, TARGET_CATEGORY_NAMES, device
)
zero_shot_time = time.time() - start_time

print(f"\nZero-shot classification completed in {zero_shot_time:.2f}s")

### 7.1 Zero-Shot Metrics

In [ ]:
def compute_metrics(y_true, y_pred, target_categories):
    """
    Compute comprehensive classification metrics.
    """
    metrics = {}
    
    # Overall metrics
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['f1_macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['f1_weighted'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    metrics['precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['recall_macro'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Per-class metrics
    report = classification_report(
        y_true, y_pred, 
        target_names=target_categories,
        output_dict=True,
        zero_division=0
    )
    
    return metrics, report

# Compute zero-shot metrics
zero_shot_metrics, zero_shot_report = compute_metrics(
    zero_shot_labels, zero_shot_preds, TARGET_CATEGORY_NAMES
)

print("\n" + "="*80)
print("ZERO-SHOT CLASSIFICATION RESULTS")
print("="*80)
print(f"\nOverall Metrics:")
print(f"  Accuracy:          {zero_shot_metrics['accuracy']:.4f}")
print(f"  F1 (Macro):        {zero_shot_metrics['f1_macro']:.4f}")
print(f"  F1 (Weighted):     {zero_shot_metrics['f1_weighted']:.4f}")
print(f"  Precision (Macro): {zero_shot_metrics['precision_macro']:.4f}")
print(f"  Recall (Macro):    {zero_shot_metrics['recall_macro']:.4f}")
print(f"  Inference Time:    {zero_shot_time:.2f}s")

print(f"\nPer-Class Metrics:")
per_class_df = pd.DataFrame(zero_shot_report).T
print(per_class_df[['precision', 'recall', 'f1-score', 'support']].head(len(TARGET_CATEGORY_NAMES)))

In [ ]:
# Confusion Matrix for Zero-Shot
cm_zero = confusion_matrix(zero_shot_labels, zero_shot_preds)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_zero, display_labels=TARGET_CATEGORY_NAMES)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Zero-Shot Classification - Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Few-Shot Classification

Fine-tune CLIP với một số lượng nhỏ samples (k-shot per class)

In [ ]:
class CLIPClassifier(nn.Module):
    """
    CLIP với classification head cho few-shot learning.
    """
    def __init__(self, clip_model, num_classes, freeze_backbone=True):
        super().__init__()
        self.clip = clip_model
        
        # Freeze CLIP backbone if specified
        if freeze_backbone:
            for param in self.clip.parameters():
                param.requires_grad = False
        
        # Classification head
        hidden_size = self.clip.config.projection_dim
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, input_ids, attention_mask, pixel_values):
        # Get CLIP embeddings
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )
        
        # Concatenate image and text features
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds
        combined = torch.cat([image_embeds, text_embeds], dim=1)
        
        # Classification
        logits = self.classifier(combined)
        return logits

In [ ]:
def train_few_shot(model, train_loader, val_loader, epochs=10, lr=1e-4, device='cuda'):
    """
    Train few-shot classifier.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    
    best_val_acc = 0.0
    best_model_state = None
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask, pixel_values)
            loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        train_acc = train_correct / train_total if train_total > 0 else 0
        avg_train_loss = train_loss / len(train_loader) if len(train_loader) > 0 else 0
        
        # Validation
        val_acc = evaluate_few_shot(model, val_loader, device)
        
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {avg_train_loss:.4f}, "
              f"Train Acc: {train_acc:.4f}, "
              f"Val Acc: {val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, history, best_val_acc

def evaluate_few_shot(model, data_loader, device):
    """
    Evaluate few-shot classifier.
    """
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask, pixel_values)
            preds = logits.argmax(dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return correct / total if total > 0 else 0

def predict_few_shot(model, data_loader, device):
    """
    Get predictions from few-shot classifier.
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask, pixel_values)
            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

In [ ]:
# Train few-shot models for different k values
few_shot_results = {}

for k in K_SHOTS:
    print("\n" + "="*80)
    print(f"FEW-SHOT CLASSIFICATION: {k}-SHOT")
    print("="*80)
    
    # Create datasets with custom collate function
    train_dataset = MultimodalDataset(few_shot_datasets[k], clip_processor, TARGET_CATEGORY_NAMES)
    train_loader = DataLoader(
        train_dataset, 
        batch_size=min(8, len(train_dataset)), 
        shuffle=True, 
        collate_fn=collate_with_processor
    )
    
    print(f"\nTraining with {len(train_dataset)} samples ({k} per class)")
    
    # Create model
    model = CLIPClassifier(
        clip_model, 
        num_classes=len(TARGET_CATEGORY_NAMES),
        freeze_backbone=True
    ).to(device)
    
    # Train
    start_time = time.time()
    trained_model, history, best_val_acc = train_few_shot(
        model, 
        train_loader, 
        test_loader,
        epochs=20 if k <= 5 else 10,
        lr=1e-4 if k <= 5 else 5e-5,
        device=device
    )
    training_time = time.time() - start_time
    
    # Evaluate on test set
    start_time = time.time()
    preds, labels, probs = predict_few_shot(trained_model, test_loader, device)
    inference_time = time.time() - start_time
    
    # Compute metrics
    metrics, report = compute_metrics(labels, preds, TARGET_CATEGORY_NAMES)
    
    # Store results
    few_shot_results[k] = {
        'model': trained_model,
        'history': history,
        'preds': preds,
        'labels': labels,
        'probs': probs,
        'metrics': metrics,
        'report': report,
        'training_time': training_time,
        'inference_time': inference_time
    }
    
    print(f"\n{k}-Shot Results:")
    print(f"  Accuracy:          {metrics['accuracy']:.4f}")
    print(f"  F1 (Macro):        {metrics['f1_macro']:.4f}")
    print(f"  F1 (Weighted):     {metrics['f1_weighted']:.4f}")
    print(f"  Training Time:     {training_time:.2f}s")
    print(f"  Inference Time:    {inference_time:.2f}s")

## 9. Comprehensive Comparison: Zero-Shot vs Few-Shot

In [ ]:
# Create comparison dataframe
comparison_data = []

# Zero-shot
comparison_data.append({
    'Method': 'Zero-Shot',
    'K': 0,
    'Accuracy': zero_shot_metrics['accuracy'],
    'F1 (Macro)': zero_shot_metrics['f1_macro'],
    'F1 (Weighted)': zero_shot_metrics['f1_weighted'],
    'Precision': zero_shot_metrics['precision_macro'],
    'Recall': zero_shot_metrics['recall_macro'],
    'Training Time (s)': 0,
    'Inference Time (s)': zero_shot_time
})

# Few-shot
for k, results in few_shot_results.items():
    comparison_data.append({
        'Method': f'{k}-Shot',
        'K': k,
        'Accuracy': results['metrics']['accuracy'],
        'F1 (Macro)': results['metrics']['f1_macro'],
        'F1 (Weighted)': results['metrics']['f1_weighted'],
        'Precision': results['metrics']['precision_macro'],
        'Recall': results['metrics']['recall_macro'],
        'Training Time (s)': results['training_time'],
        'Inference Time (s)': results['inference_time']
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("OVERALL COMPARISON: ZERO-SHOT vs FEW-SHOT")
print("="*80)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Accuracy comparison
axes[0, 0].plot(comparison_df['K'], comparison_df['Accuracy'], marker='o', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Number of Shots (K)', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy vs K-Shot', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticks(comparison_df['K'])

# F1 Score comparison
axes[0, 1].plot(comparison_df['K'], comparison_df['F1 (Macro)'], marker='s', linewidth=2, markersize=8, label='F1 Macro')
axes[0, 1].plot(comparison_df['K'], comparison_df['F1 (Weighted)'], marker='^', linewidth=2, markersize=8, label='F1 Weighted')
axes[0, 1].set_xlabel('Number of Shots (K)', fontsize=12)
axes[0, 1].set_ylabel('F1 Score', fontsize=12)
axes[0, 1].set_title('F1 Score vs K-Shot', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticks(comparison_df['K'])

# Precision & Recall
axes[1, 0].plot(comparison_df['K'], comparison_df['Precision'], marker='D', linewidth=2, markersize=8, label='Precision')
axes[1, 0].plot(comparison_df['K'], comparison_df['Recall'], marker='v', linewidth=2, markersize=8, label='Recall')
axes[1, 0].set_xlabel('Number of Shots (K)', fontsize=12)
axes[1, 0].set_ylabel('Score', fontsize=12)
axes[1, 0].set_title('Precision & Recall vs K-Shot', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xticks(comparison_df['K'])

# Training Time
axes[1, 1].bar(comparison_df['Method'], comparison_df['Training Time (s)'], color='skyblue', edgecolor='black')
axes[1, 1].set_xlabel('Method', fontsize=12)
axes[1, 1].set_ylabel('Training Time (s)', fontsize=12)
axes[1, 1].set_title('Training Time Comparison', fontsize=14, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Per-Class Performance Analysis

In [ ]:
# Compare per-class F1 scores
per_class_comparison = pd.DataFrame({
    'Category': TARGET_CATEGORY_NAMES
})

# Zero-shot
per_class_comparison['Zero-Shot F1'] = [zero_shot_report[cat]['f1-score'] for cat in TARGET_CATEGORY_NAMES]

# Few-shot
for k in K_SHOTS:
    report = few_shot_results[k]['report']
    per_class_comparison[f'{k}-Shot F1'] = [report[cat]['f1-score'] for cat in TARGET_CATEGORY_NAMES]

print("\nPer-Class F1 Score Comparison:")
print(per_class_comparison.to_string(index=False))

# Visualize per-class performance
fig, ax = plt.subplots(figsize=(14, 8))
x = np.arange(len(TARGET_CATEGORY_NAMES))
width = 0.15

ax.bar(x - 2*width, per_class_comparison['Zero-Shot F1'], width, label='Zero-Shot', alpha=0.8)
for i, k in enumerate(K_SHOTS):
    ax.bar(x + (i-1)*width, per_class_comparison[f'{k}-Shot F1'], width, label=f'{k}-Shot', alpha=0.8)

ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1 Score Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(TARGET_CATEGORY_NAMES, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 11. Confusion Matrices Comparison

In [ ]:
# Plot confusion matrices for best few-shot result
best_k = max(K_SHOTS, key=lambda k: few_shot_results[k]['metrics']['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Zero-shot
cm_zero = confusion_matrix(zero_shot_labels, zero_shot_preds)
disp_zero = ConfusionMatrixDisplay(confusion_matrix=cm_zero, display_labels=TARGET_CATEGORY_NAMES)
disp_zero.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title(f'Zero-Shot (Acc: {zero_shot_metrics["accuracy"]:.3f})', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Best few-shot
cm_few = confusion_matrix(few_shot_results[best_k]['labels'], few_shot_results[best_k]['preds'])
disp_few = ConfusionMatrixDisplay(confusion_matrix=cm_few, display_labels=TARGET_CATEGORY_NAMES)
disp_few.plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title(f'{best_k}-Shot (Acc: {few_shot_results[best_k]["metrics"]["accuracy"]:.3f})', 
                  fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 12. Summary và Kết luận

In [ ]:
print("\n" + "="*80)
print("SUMMARY AND CONCLUSIONS")
print("="*80)

print("\n1. Dataset:")
print(f"   - Name: COCO 2017 (Multimodal: Image + Caption)")
print(f"   - Source: {'Kaggle' if IS_KAGGLE else 'HuggingFace (fallback)'}")
print(f"   - Classes: {len(TARGET_CATEGORY_NAMES)} ({', '.join(TARGET_CATEGORY_NAMES)})")
print(f"   - Test samples: {len(test_df)}")

print("\n2. Model:")
print(f"   - Architecture: CLIP (openai/clip-vit-base-patch32)")
print(f"   - Parameters: {total_params:,}")

print("\n3. Performance Comparison:")
print(f"\n   Zero-Shot:")
print(f"   - Accuracy: {zero_shot_metrics['accuracy']:.4f}")
print(f"   - F1 (Macro): {zero_shot_metrics['f1_macro']:.4f}")
print(f"   - Training: None required")
print(f"   - Inference: {zero_shot_time:.2f}s")

for k in K_SHOTS:
    metrics = few_shot_results[k]['metrics']
    print(f"\n   {k}-Shot:")
    print(f"   - Accuracy: {metrics['accuracy']:.4f} "
          f"(+{(metrics['accuracy'] - zero_shot_metrics['accuracy'])*100:.2f}%)")
    print(f"   - F1 (Macro): {metrics['f1_macro']:.4f} "
          f"(+{(metrics['f1_macro'] - zero_shot_metrics['f1_macro'])*100:.2f}%)")
    print(f"   - Training: {few_shot_results[k]['training_time']:.2f}s")
    print(f"   - Inference: {few_shot_results[k]['inference_time']:.2f}s")

print("\n4. Key Findings:")
print(f"   - Zero-shot CLIP shows strong baseline performance without any training")
print(f"   - Few-shot learning consistently improves over zero-shot")
print(f"   - Performance gain diminishes as K increases (law of diminishing returns)")
print(f"   - Best K-shot: {best_k} with accuracy {few_shot_results[best_k]['metrics']['accuracy']:.4f}")
improvement = (few_shot_results[best_k]['metrics']['accuracy'] - zero_shot_metrics['accuracy']) * 100
print(f"   - Improvement over zero-shot: +{improvement:.2f}%")

print("\n5. Recommendations:")
if improvement < 5:
    print("   - Zero-shot CLIP is highly effective for this task")
    print("   - Consider zero-shot for deployment to save training costs")
elif improvement < 10:
    print("   - Moderate improvement with few-shot learning")
    print("   - Use few-shot if higher accuracy is required")
else:
    print("   - Significant improvement with few-shot learning")
    print("   - Few-shot approach is recommended for this task")

print("\n" + "="*80)

## 13. Save Results

In [ ]:
# Save comparison results to CSV
comparison_df.to_csv('multimodal_comparison_results.csv', index=False)
print("Saved comparison results to 'multimodal_comparison_results.csv'")

# Save per-class comparison
per_class_comparison.to_csv('multimodal_per_class_results.csv', index=False)
print("Saved per-class results to 'multimodal_per_class_results.csv'")

print("\nAll results saved successfully!")